In [ ]:
!python -m pip install --upgrade pip setuptools wheel build pip-tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
import sys
import subprocess
import pkgutil

required = [
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("datasets", "datasets"),
    ("transformers", "transformers"),
    ("snorkel", "snorkel"),
    ("wandb", "wandb"),
    ("tqdm", "tqdm"),
    ("reportlab", "reportlab"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("scikit-learn", "sklearn"),
]

missing = []
for pkg_name, import_name in required:
    if pkgutil.find_loader(import_name) is None:
        missing.append(pkg_name)

if missing:
    print("Missing packages detected:", missing)
    print("Installing missing packages.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages appear installed.")

/tmp/ipython-input-4077807953.py:21: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader(import_name) is None:


Missing packages detected: ['snorkel', 'reportlab']
Installing missing packages.


In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 142502032 (ir2023) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# CIFAR sequential training (ResNet18) — small helper experiment
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import os # Import os for creating directories

PROJECT_CIFAR = "cifar-sequential"

# Dataloaders
def get_dataloaders(name, batch_size=128, num_workers=4):
    if name == "CIFAR10":
        mean = [0.4914, 0.4822, 0.4465]
        std = [0.2470, 0.2435, 0.2616]
    else:
        mean = [0.5071, 0.4867, 0.4408]
        std = [0.2675, 0.2565, 0.2761]

    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    if name == "CIFAR10":
        trainset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
        testset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)
        num_classes = 10
    else:
        trainset = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform_train)
        testset = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform_test)
        num_classes = 100

    trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return trainloader, testloader, num_classes

# Build adapted ResNet18 for CIFAR

def build_resnet18(num_classes, device):
    model = models.resnet18(pretrained=False)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

# Train & eval

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += X.size(0)
    return running_loss / total, 100.0 * correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = criterion(out, y)
            running_loss += loss.item() * X.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += X.size(0)
    return running_loss / total, 100.0 * correct / total

# Experiment runner

def run_sequential_experiment(first, second, epochs=10, batch_size=128, device=None, run_name_suffix="run"):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    run = wandb.init(project=PROJECT_CIFAR, name=f"{first}_then_{second}_{run_name_suffix}", reinit=True)

    # Phase 1
    train1, test1, num_cls1 = get_dataloaders(first, batch_size=batch_size)
    model = build_resnet18(num_cls1, device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[int(epochs*0.6), int(epochs*0.8)], gamma=0.1)
    print(f"Training phase 1: {first} for {epochs} epochs on {device}")
    for ep in range(epochs):
        tr_loss, tr_acc = train_epoch(model, train1, optimizer, criterion, device)
        val_loss, val_acc = eval_epoch(model, test1, criterion, device)
        scheduler.step()
        wandb.log({"phase": first, "epoch": ep, "train_loss": tr_loss, "train_acc": tr_acc, "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{first}] epoch {ep}: train_acc={tr_acc:.2f} val_acc={val_acc:.2f}")

    # Save checkpoint
    ckpt_dir = "checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt1 = f"{ckpt_dir}/{first}_model.pth"
    torch.save(model.state_dict(), ckpt1)
    wandb.save(ckpt1)

    # Phase 2: adjust final fc if class count differs
    train2, test2, num_cls2 = get_dataloaders(second, batch_size=batch_size)
    if num_cls2 != num_cls1:
        model.fc = nn.Linear(model.fc.in_features, num_cls2).to(device)
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
        scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[int(epochs*0.6), int(epochs*0.8)], gamma=0.1)

    print(f"Training phase 2: {second} for {epochs} epochs on {device}")
    for ep in range(epochs):
        tr_loss, tr_acc = train_epoch(model, train2, optimizer, criterion, device)
        val_loss, val_acc = eval_epoch(model, test2, criterion, device)
        scheduler.step()
        wandb.log({"phase": second, "epoch": epochs + ep, "train_loss": tr_loss, "train_acc": tr_acc, "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{second}] epoch {ep}: train_acc={tr_acc:.2f} val_acc={val_acc:.2f}")

    ckpt2 = f"{ckpt_dir}/{second}_after_{first}_model.pth"
    torch.save(model.state_dict(), ckpt2)
    wandb.save(ckpt2)
    wandb.finish()

# Experiment A: CIFAR-100 then CIFAR-10 (100 epochs each — heavy)
run_sequential_experiment(
    first="CIFAR100",
    second="CIFAR10",
    epochs=100,
    batch_size=128,
    device="cuda" if torch.cuda.is_available() else "cpu",
    run_name_suffix="expA"
)

# Experiment B: CIFAR-10 then CIFAR-100 (100 epochs each)
run_sequential_experiment(
    first="CIFAR10",
    second="CIFAR100",
    epochs=100,
    batch_size=128,
    device="cuda" if torch.cuda.is_available() else "cpu",
    run_name_suffix="expB"
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


100%|██████████| 169M/169M [00:04<00:00, 39.9MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warn

Training phase 1: CIFAR100 for 100 epochs on cuda
[CIFAR100] epoch 0: train_acc=9.89 val_acc=15.33
[CIFAR100] epoch 1: train_acc=20.26 val_acc=23.33
[CIFAR100] epoch 2: train_acc=30.39 val_acc=30.95
[CIFAR100] epoch 3: train_acc=39.41 val_acc=36.86
[CIFAR100] epoch 4: train_acc=45.44 val_acc=39.58
[CIFAR100] epoch 5: train_acc=49.95 val_acc=46.73
[CIFAR100] epoch 6: train_acc=53.13 val_acc=46.83
[CIFAR100] epoch 7: train_acc=55.21 val_acc=45.49
[CIFAR100] epoch 8: train_acc=56.82 val_acc=50.19
[CIFAR100] epoch 9: train_acc=58.08 val_acc=51.87
[CIFAR100] epoch 10: train_acc=59.85 val_acc=49.32
[CIFAR100] epoch 11: train_acc=60.64 val_acc=50.08
[CIFAR100] epoch 12: train_acc=61.76 val_acc=55.12
[CIFAR100] epoch 13: train_acc=62.15 val_acc=56.41
[CIFAR100] epoch 14: train_acc=62.89 val_acc=53.46
[CIFAR100] epoch 15: train_acc=63.52 val_acc=54.69
[CIFAR100] epoch 16: train_acc=64.49 val_acc=53.70
[CIFAR100] epoch 17: train_acc=65.24 val_acc=55.21
[CIFAR100] epoch 18: train_acc=64.97 val_ac

100%|██████████| 170M/170M [00:03<00:00, 43.6MB/s]


Training phase 2: CIFAR10 for 100 epochs on cuda
[CIFAR10] epoch 0: train_acc=82.56 val_acc=88.00
[CIFAR10] epoch 1: train_acc=90.58 val_acc=90.16
[CIFAR10] epoch 2: train_acc=92.43 val_acc=90.60
[CIFAR10] epoch 3: train_acc=93.96 val_acc=91.04
[CIFAR10] epoch 4: train_acc=94.85 val_acc=91.50
[CIFAR10] epoch 5: train_acc=95.52 val_acc=91.64
[CIFAR10] epoch 6: train_acc=96.19 val_acc=90.94
[CIFAR10] epoch 7: train_acc=96.60 val_acc=91.65
[CIFAR10] epoch 8: train_acc=96.89 val_acc=91.57
[CIFAR10] epoch 9: train_acc=97.01 val_acc=91.65
[CIFAR10] epoch 10: train_acc=97.33 val_acc=91.87
[CIFAR10] epoch 11: train_acc=97.65 val_acc=92.07
[CIFAR10] epoch 12: train_acc=97.55 val_acc=90.91
[CIFAR10] epoch 13: train_acc=97.76 val_acc=91.99
[CIFAR10] epoch 14: train_acc=97.98 val_acc=91.99
[CIFAR10] epoch 15: train_acc=98.05 val_acc=92.06
[CIFAR10] epoch 16: train_acc=98.15 val_acc=92.24
[CIFAR10] epoch 17: train_acc=98.06 val_acc=91.94
[CIFAR10] epoch 18: train_acc=98.29 val_acc=92.22
[CIFAR10] e

...

epoch,▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇██████
train_acc,▁▃▄▄▅▅▅▅▅▅▅▅▅▅▇██████████▇██████████████
train_loss,█▇▅▅▅▅▄▄▄▄▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▂▃▃▃▃▃▃▃▃▃▃▃▄▃▅▆▆▆▆▆▆▇█████████████████
val_loss,█▅▄▄▄▄▄▄▄▄▅▄▄▄▂▃▃▃▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,199
phase,CIFAR10
train_acc,99.994
train_loss,0.00251
val_acc,95.1
val_loss,0.18423


Training phase 1: CIFAR10 for 100 epochs on cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


[CIFAR10] epoch 0: train_acc=31.89 val_acc=43.20
[CIFAR10] epoch 1: train_acc=51.51 val_acc=58.33
[CIFAR10] epoch 2: train_acc=62.69 val_acc=65.76
[CIFAR10] epoch 3: train_acc=70.67 val_acc=74.35
[CIFAR10] epoch 4: train_acc=75.94 val_acc=75.20
[CIFAR10] epoch 5: train_acc=78.98 val_acc=73.59
[CIFAR10] epoch 6: train_acc=81.07 val_acc=75.74
[CIFAR10] epoch 7: train_acc=82.24 val_acc=77.25
[CIFAR10] epoch 8: train_acc=82.99 val_acc=81.69
[CIFAR10] epoch 9: train_acc=83.45 val_acc=66.49
[CIFAR10] epoch 10: train_acc=84.15 val_acc=79.32
[CIFAR10] epoch 11: train_acc=84.83 val_acc=80.44
[CIFAR10] epoch 12: train_acc=84.78 val_acc=84.51
[CIFAR10] epoch 13: train_acc=85.61 val_acc=74.63
[CIFAR10] epoch 14: train_acc=85.81 val_acc=81.76
[CIFAR10] epoch 15: train_acc=86.15 val_acc=84.95
[CIFAR10] epoch 16: train_acc=86.26 val_acc=81.22
[CIFAR10] epoch 17: train_acc=86.67 val_acc=83.74
[CIFAR10] epoch 18: train_acc=86.44 val_acc=82.07
[CIFAR10] epoch 19: train_acc=86.58 val_acc=85.37
[CIFAR10] 

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇███
train_acc,▁▅▆▆▆▆▆▆▆▆▆▇██████▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇███████
train_loss,▇▅▃▃▃▃▃▃▃▃▃▃▃▃▃▂▁▁▁▁▁▁▁█▅▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁
val_acc,▃▄▅▆▅▆▆▆▅▅████████▁▂▂▂▂▂▂▂▂▂▂▂▃▄▄▄▄▄▄▄▄▄
val_loss,▅▃▂▂▄▃▃▃▃▃▂▂▃▁▁▁▁▁▁▁█▇▇▇▆▇▇▇▇▇▇▇▅▅▅▅▅▅▅▅
epoch,199
phase,CIFAR100
train_acc,99.922
train_loss,0.01158
val_acc,77
val_loss,0.98289
